# Data Preprocessing: Armenian Participle Extraction & Alignment

**Purpose:** Extracts valid sentences containing specific participial suffixes (`-լով` and `-ած`) from a massive raw Armenian corpus, applying strict formatting and noise-reduction filters. Additionally, aligns unpunctuated text with human-annotated "gold" text for the evaluation and fine-tuning datasets.

**Inputs:**
- Raw text corpus (`hy.txt`).
- Raw unpunctuated and gold-standard text pairs (`Shtemaran/aligned/` and `Shtemaran/gold/`).

**Outputs:**
- Filtered corpus subsets categorized by suffix (`corpus_lov.txt`, `corpus_ac.txt`, `corpus_both.txt`).
- Character-aligned, paired evaluation sentences (`merged_np_lov_ac.txt`, `merged_gold_lov_ac.txt`).

**Estimated Runtime:** ~15-20 minutes depending on I/O speed for the multi-gigabyte corpus.

## Setup and Configuration

In [ ]:
import os
import re
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR = Path("./data")
RAW_CORPUS_PATH = DATA_DIR / "raw" / "hy.txt"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LOV_OUT = PROCESSED_DIR / "corpus_lov.txt"
AC_OUT = PROCESSED_DIR / "corpus_ac.txt"
BOTH_OUT = PROCESSED_DIR / "corpus_both.txt"

SHTEMARAN_DIR = DATA_DIR / "Shtemaran"
ALIGNED_DIR = SHTEMARAN_DIR / "aligned"
GOLD_DIR = SHTEMARAN_DIR / "gold"
MERGED_NP_OUT = SHTEMARAN_DIR / "merged_np_lov_ac.txt"
MERGED_GOLD_OUT = SHTEMARAN_DIR / "merged_gold_lov_ac.txt"

## Phase 1: Corpus Participle Extraction

In [ ]:
def process_and_split_sentences(file_path, out_lov, out_ac, out_both):
    """Parses a massive text corpus, separates sentences using punctuation heuristics,
    filters out URLs/references, and categorizes sentences by participial suffix.

    Args:
        file_path (Path): Path to the raw corpus text file.
        out_lov (Path): Output path for sentences containing only the -լով suffix.
        out_ac (Path): Output path for sentences containing only the -ած suffix.
        out_both (Path): Output path for sentences containing both suffixes.
    """
    if not file_path.exists():
        return

    wiki_ref_pattern = re.compile(r'\[\d+\]')
    split_pattern = re.compile(r'(\.\.\.|(?:[:.](?=\s))|(?:[:.]$))')
    pat_lov = re.compile(r'\w+լով\b')
    pat_ac = re.compile(r'\w+ած\b')
    
    total_valid_strict_sentences = 0
    count_lov, count_ac, count_both = 0, 0, 0
    total_size = os.path.getsize(file_path)

    with open(file_path, 'r', encoding='utf-8', errors='ignore') as infile, \
         open(out_lov, 'w', encoding='utf-8') as f_lov, \
         open(out_ac, 'w', encoding='utf-8') as f_ac, \
         open(out_both, 'w', encoding='utf-8') as f_both, \
         tqdm(total=total_size, unit='B', unit_scale=True, desc="Processing Corpus") as pbar:
        
        for line in infile:
            pbar.update(len(line.encode('utf-8')))
            line = line.strip()
            if not line:
                continue

            line = wiki_ref_pattern.sub('', line)
            parts = split_pattern.split(line)
            
            for i in range(0, len(parts) - 1, 2):
                sentence_body = parts[i]
                delimiter = parts[i+1]
                full_sentence = (sentence_body + delimiter).strip()
                
                if len(full_sentence) < 2:
                    continue
                    
                if "https" in full_sentence:
                    continue
                if any(ext in full_sentence for ext in [".com", ".am", ".net", ".org"]):
                    continue

                total_valid_strict_sentences += 1
                
                has_lov = bool(pat_lov.search(full_sentence))
                has_ac = bool(pat_ac.search(full_sentence))
                
                if has_lov and has_ac:
                    count_both += 1
                    f_both.write(f"{count_both}. {full_sentence}\n")
                elif has_ac:
                    count_ac += 1
                    f_ac.write(f"{count_ac}. {full_sentence}\n")
                elif has_lov:
                    count_lov += 1
                    f_lov.write(f"{count_lov}. {full_sentence}\n")

process_and_split_sentences(RAW_CORPUS_PATH, LOV_OUT, AC_OUT, BOTH_OUT)

## Phase 2: Gold Data Alignment and Normalization

In [ ]:
def fix_ocr(text):
    """Resolves common optical character recognition artifacts in the text.

    Args:
        text (str): The raw text string.

    Returns:
        str: The normalized text string.
    """
    text = text.replace('6', 'բ')
    text = text.replace('2', 'ե')
    text = text.replace('くらい', 'բողջ')
    text = text.replace('~', ' ')
    return text

def normalize_text(text):
    """Applies OCR fixes, removes carriage returns, and collapses whitespace.

    Args:
        text (str): The string to normalize.

    Returns:
        str: The fully cleaned string.
    """
    text = fix_ocr(text)
    text = text.replace('\n', ' ').replace('\r', ' ')
    while '  ' in text:
        text = text.replace('  ', ' ')
    return text.strip()

def align_and_merge_shtemaran(aligned_dir, gold_dir, out_np, out_gold, count=99):
    """Aligns unpunctuated text files with annotated gold equivalents by comparing 
    alphanumeric sequences, skipping punctuation gaps to ensure structural parity.

    Args:
        aligned_dir (Path): Directory containing unpunctuated input text.
        gold_dir (Path): Directory containing correctly punctuated gold text.
        out_np (Path): Output path for merged unpunctuated sentences.
        out_gold (Path): Output path for merged gold sentences.
        count (int): Number of file pairs to process.
    """
    if not aligned_dir.exists() or not gold_dir.exists():
        return

    wiki_ref_pattern = re.compile(r'\[\d+\]')
    split_pattern = re.compile(r'(\.\.\.|…|(?:[։.:!?](?=\s))|(?:[։.:!?]$))')
    pat_lov = re.compile(r'\w+լով\b')
    pat_ac = re.compile(r'\w+ած\b')
    inserted_punct = {',', '՝', '-', '–', ':', '։', '.', '!', '?', '"', "'", '(', ')', '[', ']', ';'}

    global_counter = 1
    with open(out_np, 'w', encoding='utf-8') as outnp, \
         open(out_gold, 'w', encoding='utf-8') as outgold:
        
        for i in tqdm(range(1, count + 1), desc="Aligning Pairs"):
            np_path = aligned_dir / f"{i}_np.txt"
            gold_path = gold_dir / f"{i}.txt"
            
            if not np_path.exists() or not gold_path.exists():
                continue

            with open(np_path, 'r', encoding='utf-8', errors='ignore') as npfile:
                np_text = normalize_text(wiki_ref_pattern.sub('', npfile.read()))
            with open(gold_path, 'r', encoding='utf-8', errors='ignore') as goldfile:
                gold_text = normalize_text(wiki_ref_pattern.sub('', goldfile.read()))

            parts = split_pattern.split(gold_text)
            np_ptr = 0
            
            for j in range(0, len(parts) - 1, 2):
                content = parts[j]
                delim = parts[j+1]
                gold_sentence = (content + delim).strip()
                
                if len(gold_sentence) < 2 or "https" in gold_sentence:
                    continue
                if any(x in gold_sentence for x in [".com", ".am", ".net", ".org"]):
                    continue

                gold_ptr = 0
                np_start = np_ptr
                mismatch = False
                
                while gold_ptr < len(gold_sentence):
                    c = gold_sentence[gold_ptr]
                    gold_ptr += 1
                    if c in inserted_punct or c.isspace():
                        continue
                    
                    while np_ptr < len(np_text) and (np_text[np_ptr] in inserted_punct or np_text[np_ptr].isspace()):
                        np_ptr += 1
                    
                    if np_ptr < len(np_text) and np_text[np_ptr] == c:
                        np_ptr += 1
                    else:
                        mismatch = True
                        break

                if mismatch:
                    continue

                np_sentence = np_text[np_start:np_ptr].strip()

                if pat_lov.search(gold_sentence) or pat_ac.search(gold_sentence):
                    outnp.write(f"{global_counter}-{i}. {np_sentence}\n")
                    outgold.write(f"{global_counter}-{i}. {gold_sentence}\n")
                    global_counter += 1

align_and_merge_shtemaran(ALIGNED_DIR, GOLD_DIR, MERGED_NP_OUT, MERGED_GOLD_OUT)